In [ ]:
!pip -q install scikit-learn imbalanced-learn joblib pandas numpy

import numpy as np
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.metrics import (confusion_matrix, classification_report,
                             accuracy_score, precision_score, recall_score, f1_score)

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

In [ ]:
import pandas as pd

df = pd.read_csv("/content/gym_churn_camelcase.csv")
df.head()


,idClient,gender,nearLocation,partner,promoFriends,phone,contractPeriod,groupVisits,age,avgAdditionalChargesTotal,monthToEndContract,lifetime,avgClassFrequencyTotal,avgClassFrequencyCurrentMonth,churn
0,1,1,1,1,1,0,6,1,29,14.227470,5,3,0.020398,0.000000,0
1,2,0,1,0,0,1,12,1,31,113.202938,12,7,1.922936,1.910244,0
2,3,0,1,1,0,1,1,0,28,129.448479,1,2,1.859098,1.736502,0
3,4,0,1,1,1,1,12,1,33,62.669863,12,2,3.205633,3.357215,0
4,5,1,1,1,1,1,1,0,26,198.362265,1,3,1.113884,1.120078,0


In [ ]:
df.shape, df.columns.tolist()


((4000, 15),
 ['idClient',
  'gender',
  'nearLocation',
  'partner',
  'promoFriends',
  'phone',
  'contractPeriod',
  'groupVisits',
  'age',
  'avgAdditionalChargesTotal',
  'monthToEndContract',
  'lifetime',
  'avgClassFrequencyTotal',
  'avgClassFrequencyCurrentMonth',
  'churn'])

In [ ]:
TARGET_COL = "churn"


In [ ]:
df[TARGET_COL].value_counts(normalize=True)


,proportion
churn,
0,0.73475
1,0.26525


In [ ]:
y = df[TARGET_COL].astype(int)
X = df.drop(columns=[TARGET_COL])

print("X shape:", X.shape)
print("Distribución churn:\n", y.value_counts(normalize=True))


X shape: (4000, 14)
Distribución churn:
 churn
0    0.73475
1    0.26525
Name: proportion, dtype: float64


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Churn Train:", y_train.mean())
print("Churn Test :", y_test.mean())


Churn Train: 0.2653125
Churn Test : 0.265


In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import confusion_matrix, classification_report

dummy = DummyClassifier(strategy="most_frequent", random_state=42)
dummy.fit(X_train, y_train)

y_pred_dummy = dummy.predict(X_test)

print("Confusion Matrix (Dummy):\n", confusion_matrix(y_test, y_pred_dummy))
print("\nClassification Report (Dummy):\n")
print(classification_report(y_test, y_pred_dummy, zero_division=0))


Confusion Matrix (Dummy):
 [[588   0]
 [212   0]]

Classification Report (Dummy):

              precision    recall  f1-score   support

           0       0.73      1.00      0.85       588
           1       0.00      0.00      0.00       212

    accuracy                           0.73       800
   macro avg       0.37      0.50      0.42       800
weighted avg       0.54      0.73      0.62       800



In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

metrics_dummy = {
    "Accuracy": accuracy_score(y_test, y_pred_dummy),
    "Precision": precision_score(y_test, y_pred_dummy, zero_division=0),
    "Recall": recall_score(y_test, y_pred_dummy, zero_division=0),
    "F1": f1_score(y_test, y_pred_dummy, zero_division=0),
}

metrics_dummy


{'Accuracy': 0.735, 'Precision': 0.0, 'Recall': 0.0, 'F1': 0.0}

In [ ]:
import joblib

joblib.dump(dummy, "dummy_baseline.joblib")
print("✅ dummy_baseline.joblib guardado")


✅ dummy_baseline.joblib guardado


In [ ]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)
X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)

print("Distribución original:\n", y_train.value_counts())
print("\nDistribución balanceada:\n", y_train_bal.value_counts())


Distribución original:
 churn
0    2351
1     849
Name: count, dtype: int64

Distribución balanceada:
 churn
0    2351
1    2351
Name: count, dtype: int64


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

scaler = StandardScaler()

X_train_bal_scaled = scaler.fit_transform(X_train_bal)
X_test_scaled = scaler.transform(X_test)

model = LogisticRegression(max_iter=2000, random_state=42)
model.fit(X_train_bal_scaled, y_train_bal)



LogisticRegression(max_iter=2000, random_state=42)

In [ ]:
# PREDICCIÓN CORRECTA
y_pred = model.predict(X_test_scaled)
y_proba = model.predict_proba(X_test_scaled)[:, 1]

print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred, zero_division=0))



Confusion Matrix:
 [[558  30]
 [ 27 185]]

Classification Report:

              precision    recall  f1-score   support

           0       0.95      0.95      0.95       588
           1       0.86      0.87      0.87       212

    accuracy                           0.93       800
   macro avg       0.91      0.91      0.91       800
weighted avg       0.93      0.93      0.93       800



In [ ]:
metrics_model = {
    "Accuracy": accuracy_score(y_test, y_pred),
    "Precision": precision_score(y_test, y_pred),
    "Recall": recall_score(y_test, y_pred),
    "F1": f1_score(y_test, y_pred),
}

metrics_model


{'Accuracy': 0.92875,
 'Precision': 0.8604651162790697,
 'Recall': 0.8726415094339622,
 'F1': 0.8665105386416861}

In [ ]:
import pandas as pd
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

pipe = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("smote", SMOTE(random_state=42)),
    ("lr", LogisticRegression(max_iter=5000, random_state=42))
])

scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1"
}

cv_results = cross_validate(
    pipe,
    X_train,   # 👈 OJO: train ORIGINAL, sin SMOTE
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=-1
)

pd.DataFrame(cv_results).agg(["mean", "std"])


,fit_time,score_time,test_accuracy,test_precision,test_recall,test_f1
mean,0.080083,0.027108,0.915938,0.799584,0.911674,0.851866
std,0.040749,0.009156,0.009657,0.014850,0.026584,0.018060


In [ ]:
import pandas as pd

report = pd.DataFrame([
    {"Modelo": "Dummy", **metrics_dummy},
    {"Modelo": "LogisticRegression", **metrics_model}
])

report


,Modelo,Accuracy,Precision,Recall,F1
0,Dummy,0.73500,0.000000,0.000000,0.000000
1,LogisticRegression,0.92875,0.860465,0.872642,0.866511


In [ ]:
report.to_markdown(index=False)


'| Modelo             |   Accuracy |   Precision |   Recall |       F1 |\n|:-------------------|-----------:|------------:|---------:|---------:|\n| Dummy              |    0.735   |    0        | 0        | 0        |\n| LogisticRegression |    0.92875 |    0.860465 | 0.872642 | 0.866511 |'

In [ ]:
joblib.dump(model, "churn_model.joblib")
print("🚀 churn_model.joblib guardado")


🚀 churn_model.joblib guardado


In [ ]:
sample = X_test.iloc[[0]]

sample_scaled = scaler.transform(sample)   # 👈 clave
pred = model.predict(sample_scaled)[0]
proba = model.predict_proba(sample_scaled)[0, 1]

print("SMOKE TEST OK ✅")
print("Predicción:", pred)
print("Probabilidad:", round(proba, 4))


SMOKE TEST OK ✅
Predicción: 0
Probabilidad: 0.0031
